# PotholeSense — training a pothole detectorFine-tunes YOLOv8 on a public pothole dataset and exports weights for thePotholeSense server.**Run this on Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU).Training YOLOv8n for 60 epochs on ~1,000 images takes roughly 20–30 minutes on a T4.At the end you download `pothole_yolov8n.pt` and drop it into the project's`models/` folder — the server picks it up automatically on restart.

In [ ]:
!nvidia-smi!pip install -q ultralytics roboflow

## 1. Get the datasetThe easiest source is **Roboflow Universe**, which hosts several annotatedpothole datasets already in YOLO format.1. Make a free account at https://roboflow.com2. Open a pothole dataset, e.g. https://public.roboflow.com/object-detection/pothole3. Click **Download this Dataset** → format **YOLOv8** → **show download code**4. Paste your API key below.Pick a dataset with **at least ~700 images** and check a few annotations by eyebefore training — several public pothole sets have noisy boxes, and garbagelabels cost you far more than an extra hour of training.

In [ ]:
from roboflow import RoboflowROBOFLOW_API_KEY = ""   # <-- paste your keyrf = Roboflow(api_key=ROBOFLOW_API_KEY)project = rf.workspace("brad-dwyer").project("pothole-voxrl")dataset = project.version(1).download("yolov8")DATA_YAML = f"{dataset.location}/data.yaml"print(DATA_YAML)

### Alternative: your own dataThe strongest version of this project uses **your own dashcam footage** as anextra test set — a few hundred frames from roads you actually drive, labelledin [Roboflow](https://roboflow.com) or [CVAT](https://cvat.ai).Public datasets are mostly sunny, dry, close-range shots. UK roads inFebruary are wet, grey and full of shadows and drain covers. A model thatscores well on a public test split and then fails on your own footage is thesingle most interesting result you can report — and fixing it (more data,augmentation, hard-negative mining) is what turns this from a tutorialreproduction into a genuine piece of work.

In [ ]:
# Inspect the dataset before training - always look at your data.import yaml, glob, randomfrom PIL import Imageimport matplotlib.pyplot as pltcfg = yaml.safe_load(open(DATA_YAML))print(cfg)imgs = glob.glob(f"{dataset.location}/train/images/*.jpg")print("train images:", len(imgs))fig, axes = plt.subplots(2, 4, figsize=(16, 7))for ax, path in zip(axes.ravel(), random.sample(imgs, 8)):    ax.imshow(Image.open(path)); ax.axis("off")plt.tight_layout(); plt.show()

## 2. Train`yolov8n` (nano) is deliberate: it is small enough to later export to ONNX orTFLite and run on a phone. If you only care about server-side accuracy, try`yolov8s` and compare — that comparison is itself a result worth reporting.

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolov8n.pt")          # COCO-pretrained backboneresults = model.train(    data=DATA_YAML,    epochs=60,    imgsz=640,    batch=16,    patience=15,                    # early stop if val mAP plateaus    # Augmentation aimed at the domain gap: British roads are wet and grey.    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,    degrees=5.0,                    # slight camera roll in a windscreen cradle    translate=0.1, scale=0.5,    fliplr=0.5,    mosaic=1.0,    project="potholesense", name="yolov8n_potholes",)

## 3. Evaluate`mAP50` is the headline number, but for this application **recall matters morethan precision**: a missed pothole is a defect that never gets fixed, while afalse positive costs a council officer ten seconds looking at a photo.Report the precision/recall curve, not just one operating point, and say whichconfidence threshold you chose and why.

In [ ]:
metrics = model.val()print(f"mAP50    : {metrics.box.map50:.3f}")print(f"mAP50-95 : {metrics.box.map:.3f}")print(f"precision: {metrics.box.mp:.3f}")print(f"recall   : {metrics.box.mr:.3f}")

In [ ]:
# Confusion matrix and PR curve are written to the run directory.from IPython.display import Image as IPImage, displayimport globfor f in sorted(glob.glob("potholesense/yolov8n_potholes/*.png")):    print(f); display(IPImage(filename=f, width=640))

## 4. Compare against the classical baselineThe repo ships `app/baseline.py`, a hand-tuned OpenCV detector (CLAHE +adaptive threshold + contour shape filtering). Benchmarking against it turns"my model gets 0.7 mAP" into "my model beats a tuned classical pipeline byX points", which is a much stronger claim and shows you understand what thelearning is actually buying you.

In [ ]:
# Upload app/baseline.py to the Colab session, then:import sys, cv2, glob, numpy as npsys.path.append("/content")from baseline import detect as cv_detectdef iou(a, b):    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b    ix1, iy1 = max(ax1, bx1), max(ay1, by1)    ix2, iy2 = min(ax2, bx2), min(ay2, by2)    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)    ua = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter    return inter / ua if ua > 0 else 0.0def load_labels(label_path, w, h):    boxes = []    try:        for line in open(label_path):            _, xc, yc, bw, bh = (float(v) for v in line.split()[:5])            boxes.append(((xc-bw/2)*w, (yc-bh/2)*h, (xc+bw/2)*w, (yc+bh/2)*h))    except FileNotFoundError:        pass    return boxesdef evaluate(detector, split="valid", iou_thr=0.4, limit=None):    tp = fp = fn = 0    files = sorted(glob.glob(f"{dataset.location}/{split}/images/*.jpg"))[:limit]    for img_path in files:        img = cv2.imread(img_path)        h, w = img.shape[:2]        gt = load_labels(img_path.replace("/images/", "/labels/").rsplit(".",1)[0]+".txt", w, h)        preds = detector(img)        matched = set()        for pb in preds:            hit = next((i for i, g in enumerate(gt)                        if i not in matched and iou(pb, g) >= iou_thr), None)            if hit is None: fp += 1            else: matched.add(hit); tp += 1        fn += len(gt) - len(matched)    p = tp/(tp+fp) if tp+fp else 0    r = tp/(tp+fn) if tp+fn else 0    f1 = 2*p*r/(p+r) if p+r else 0    return {"precision": round(p,3), "recall": round(r,3), "f1": round(f1,3),            "tp": tp, "fp": fp, "fn": fn}baseline_scores = evaluate(lambda im: [b for b, c in cv_detect(im)], limit=150)yolo_scores = evaluate(    lambda im: [tuple(float(v) for v in b.xyxy[0])                for b in model.predict(im, conf=0.45, verbose=False)[0].boxes],    limit=150)print("classical CV baseline:", baseline_scores)print("YOLOv8n fine-tuned   :", yolo_scores)

## 5. Export weights`best.pt` is what the server loads. The ONNX and TFLite exports are for thenext step of the project: running the model on the phone itself, so no framesever leave the device.

In [ ]:
import shutilbest = "potholesense/yolov8n_potholes/weights/best.pt"shutil.copy(best, "pothole_yolov8n.pt")m = YOLO(best)m.export(format="onnx", imgsz=640)      # cross-platform runtime# m.export(format="tflite", imgsz=640)  # uncomment for on-device Androidfrom google.colab import filesfiles.download("pothole_yolov8n.pt")

## 6. Install into PotholeSense```bashmv ~/Downloads/pothole_yolov8n.pt  potholesense/models/python run.py --https```Check it loaded:```bashcurl -k https://localhost:8000/health# {"model": ".../models/pothole_yolov8n.pt", "using_fallback_weights": false, ...}```Then re-run the evaluation harness against the real model and compare with thenumbers in the README.